# proto-REINFORCE, part 2: Old Bessie's forward pass

proto-REINFORCE judges a table of preferences by the average return of 1,000 episodes of its policy: the forward pass.
Part 1 builds an episode and its $G_1$; here they are given, finished, in a folded cell, with part 1's made-up
probability table `my_table`. You build many episodes, their average, the table of preferences and the average
return of a table of preferences. This notebook does not need part 1.

The given functions you call: `play_episode(policy, weeks)` plays one episode of `weeks` weeks and gives its list of
rewards, and `episode_return(rewards)` gives its $G_1$.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. The luck of the
game is in the kit: your code never deals with dice.

Start with the three folded cells below: Old Bessie's kit, the functions of part 1 and the checks.

In [ ]:
#@title Old Bessie's kit (run me first) { display-mode: "form" }
# Old Bessie's kit. Run this cell first; you do not need to read it.
import random                   # Python's random numbers: random.randint(0, 3) gives 0, 1, 2 or 3
from math import exp            # e to the power: exp(0.4) is 1.4918...
import math as _kit_math

STATES = ['HEALTHY', 'WORN', 'SHAKY', 'FAILING']
ACTIONS = ['RUN', 'SERVICE', 'REPLACE']
HEALTHY, WORN, SHAKY, FAILING = 0, 1, 2, 3       # the rows of a table: the van's states
RUN, SERVICE, REPLACE = 0, 1, 2                  # the columns of a table: the actions
GAMMA = 0.9                # a week later, a dollar counts 0.9 times as much
WEEKS = 13                 # the weeks of a quarter: one episode
EPISODES = 1000            # the episodes of every average, as on page 3

# The van of the arcade, as on page 1 (bessie.js): the week's money and the probabilities of what happens
_KIT_PROFIT = [95, 72, 40, 16]          # RUN: the route's money, by state
_KIT_BREAKDOWN = [0.02, 0.08, 0.28, 0.55]    # RUN: the probability of a breakdown, by state
_KIT_TOW = 280                       # a breakdown costs this on top, and the van is FAILING next week
_KIT_WEAR = [[0.35, 0.55, 0.1], [0.3, 0.55, 0.15], [0.45, 0.55, 0.0], [0.65, 0.35, 0.0]]     # RUN without a breakdown: wears 0, 1 or 2 levels
_KIT_SHOP = 50                       # SERVICE: a week in the garage
_KIT_REPAIR = [[1.0, 0.0, 0.0], [0.05, 0.7, 0.25], [0.45, 0.5, 0.05], [0.55, 0.42, 0.03]]     # SERVICE: repairs 0, 1 or 2 levels
_KIT_NEW_VAN = 130                   # REPLACE: a new van, HEALTHY next week
_KIT_START_PREFERENCES = [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]   # page 3's preferences at random
_KIT_BEST_LINE = 244.03          # the optimal policy's average return on the luck of the loop's last update
_KIT_FIRST_LUCK = 12                 # the luck of page 3's start: B.dice(12, 1000)
_KIT_LUCK_BASE = 2000               # update i of the loop plays the luck of seed 2000 + i, as page 3's movie
_KIT_PICKS = 2                       # the seed of Python's random for the loop's picks (play sets it)


def _kit_outcomes(state, action):
    """Every outcome of a week, in the order of the page: [running probability, next state, money]."""
    out = []
    if action == RUN:
        out.append([_KIT_BREAKDOWN[state], FAILING, _KIT_PROFIT[state] - _KIT_TOW])
        for d in range(3):
            p = (1 - _KIT_BREAKDOWN[state]) * _KIT_WEAR[state][d]
            if p > 0:
                out.append([p, min(FAILING, state + d), _KIT_PROFIT[state]])
    elif action == SERVICE:
        for d in range(3):
            p = _KIT_REPAIR[state][d]
            if p > 0:
                out.append([p, max(HEALTHY, state - d), -_KIT_SHOP])
    else:
        out.append([1, HEALTHY, -_KIT_NEW_VAN])
    running = 0
    for o in out:
        running = running + o[0]
        o[0] = running
    return out


_KIT_WEEK = []
for _s in range(4):
    _KIT_WEEK.append([])
    for _a in range(3):
        _KIT_WEEK[_s].append(_kit_outcomes(_s, _a))

# The luck: one stream of numbers between 0 and 1 (Mulberry32, the generator of the arcade and of page 3)
_KIT_LUCK = {"seed": _KIT_FIRST_LUCK, "state": _KIT_FIRST_LUCK, "update": 0}


def _kit_next():
    s = (_KIT_LUCK["state"] + 0x6D2B79F5) & 0xFFFFFFFF
    _KIT_LUCK["state"] = s
    t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
    t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
    return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    return _kit_next()


def transition(state, action):
    """One week of Old Bessie, the transition dynamics of page 1. In this state, with this action, luck decides what
    happens, with the transition probabilities of page 1. Gives the van's state next week and the week's money:
        next_state, money = transition(HEALTHY, RUN)"""
    if state is Ellipsis or action is Ellipsis:
        print("⏳ transition waits for a state and an action: replace each ... with yours, then run the cell again.")
        return None, None
    try:
        outcomes = _KIT_WEEK[state][action]
    except (IndexError, TypeError):
        raise ValueError("transition needs a state from 0 to 3 and an action from 0 to 2; it got the state %r and "
                         "the action %r" % (state, action))
    u = _kit_next()
    for running, next_state, money in outcomes:
        if u < running:
            return next_state, money
    return outcomes[-1][1], outcomes[-1][2]


def same_luck():
    """The luck starts again from its beginning: the episodes that follow meet the same weeks as the episodes of every
    average before them, so two tables of preferences are compared on the same luck."""
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]


def new_luck():
    """New episodes for the next update of proto-REINFORCE: the luck starts from a new seed, 2000 plus the number of the
    update."""
    _KIT_LUCK["update"] = _KIT_LUCK["update"] + 1
    _KIT_LUCK["seed"] = _KIT_LUCK_BASE + _KIT_LUCK["update"]
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]


def _kit_set_luck(seed, update=0):
    """For the checks: the luck of this seed, from its beginning."""
    _KIT_LUCK["seed"] = seed
    _KIT_LUCK["state"] = seed
    _KIT_LUCK["update"] = update


def _kit_before_long():
    """play calls this before the loop: page 3's start, and the same picks for everyone."""
    _kit_set_luck(_KIT_FIRST_LUCK)
    random.seed(_KIT_PICKS)


def _kit_after_long():
    """play calls this after the loop: page 3's start again, for the cells above."""
    _kit_set_luck(_KIT_FIRST_LUCK)


def _kit_seed_picks(seed):
    """For the checks: Python's random from this seed."""
    random.seed(seed)


def _kit_save():
    """For the checks: the state of the luck and of Python's random."""
    return [dict(_KIT_LUCK), random.getstate()]


def _kit_restore(saved):
    """For the checks: the luck and Python's random as they were."""
    _KIT_LUCK.update(saved[0])
    random.setstate(saved[1])


def start_preferences():
    """Page 3's table of preferences picked at random, between -10 and 10: a fresh copy each time."""
    return copy_preferences(_KIT_START_PREFERENCES)


def copy_preferences(preferences):
    """A copy of a table of preferences: change the copy, and the table stays as it was."""
    copy = []
    for row in preferences:
        copy.append(list(row))
    return copy


def _kit_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _kit_is_table(table, columns=3):
    if not isinstance(table, list) or len(table) != len(STATES):
        return False
    for row in table:
        if not isinstance(row, list) or len(row) != columns:
            return False
        for x in row:
            if not _kit_is_number(x):
                return False
    return True


def _kit_short(value):
    text = repr(value).replace("Ellipsis", "...")
    return text if len(text) <= 60 else text[:57] + "..."


def _kit_print_table(table, places, mark):
    head = " " * 9
    for name in ACTIONS:
        head += "%10s" % name
    if mark:
        head += "    likeliest"
    print(head)
    for s in range(len(STATES)):
        line = "%-9s" % STATES[s]
        best = 0
        for a in range(len(ACTIONS)):
            line += "%10.*f" % (places, table[s][a])
            if table[s][a] > table[s][best]:
                best = a
        if mark:
            line += "    " + ACTIONS[best]
        print(line)


def show_preferences(preferences):
    """Print a table of preferences: a row per state, a column per action, two decimals. Silent for None (a line above
    says why)."""
    if preferences is None:
        return
    if not _kit_is_table(preferences):
        print("⏳ No table of preferences to show yet: show_preferences needs four rows of three numbers. It got %s."
              % _kit_short(preferences))
        return
    _kit_print_table(preferences, 2, False)


def show_probabilities(table):
    """Print a probability table (a policy): a row per state, a column per action, two decimals, and the likeliest
    action of each row. Silent for None (a line above says why)."""
    if table is None:
        return
    if not _kit_is_table(table):
        print("⏳ No probability table to show yet: show_probabilities needs four rows of three numbers. It got %s."
              % _kit_short(table))
        return
    _kit_print_table(table, 2, True)


def show_counts(counts):
    """Print a table of counts: a row per state, a column per action, whole numbers."""
    if counts is None:
        return
    if not _kit_is_table(counts):
        print("⏳ No counts to show yet: show_counts needs four rows of three numbers. It got %s." % _kit_short(counts))
        return
    _kit_print_table(counts, 0, False)


def show_roulette(probabilities, ball=None):
    """Print the roulette of a row of probabilities: each action's slot, starting where the slot before it ends, on a
    wheel of 50 marks from 0 to 1. With a ball, a ^ marks where it is."""
    ok = isinstance(probabilities, list) and len(probabilities) == len(ACTIONS)
    if ok:
        for x in probabilities:
            if not _kit_is_number(x) or x < 0:
                ok = False
    if not ok:
        print("⏳ No roulette to show yet: show_roulette needs a row of three probabilities. It got %s."
              % _kit_short(probabilities))
        return
    start = 0
    for a in range(len(ACTIONS)):
        end = start + probabilities[a]
        left = int(round(start * 50))
        right = int(round(end * 50))
        print("%-8s %4.2f to %4.2f  |%s%s" % (ACTIONS[a], start, end, " " * left, "#" * (right - left)))
        start = end
    print(" " * 24 + "0" + " " * 49 + "1")
    if ball is not None and _kit_is_number(ball) and 0 <= ball < 1:
        print(" " * 24 + " " * int(ball * 50) + "^ the ball, %.2f" % ball)


def _kit_dollars(x):
    """$95, -$185, $119.30: whole dollars without cents."""
    if x == int(x):
        text = "$%d" % abs(int(x))
    else:
        text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def show_week(next_state, money):
    """Print one week of the van: its state next week and the week's money. Silent when there is no week yet (a
    line above says why)."""
    if next_state is None and money is None:
        return
    if isinstance(next_state, bool) or not isinstance(next_state, int) or not 0 <= next_state < len(STATES) or (
            not _kit_is_number(money)):
        print("⏳ No week to show yet: show_week needs a state and the week's money. It got %s and %s."
              % (_kit_short(next_state), _kit_short(money)))
        return
    print("Next week the van is %s. The week's money: %s." % (STATES[next_state], _kit_dollars(money)))


def show_money(label, money):
    """Print an amount of money, or a list of amounts, after a label: show_money("G1:", 154.4) prints G1: $154.40.
    A list longer than an episode shows its first WEEKS. Silent when there is nothing to show yet (a line above says
    why)."""
    if money is None:
        return
    if isinstance(money, list):
        parts = []
        for k in range(len(money)):
            if k == WEEKS:
                parts.append("... (%s in all)" % format(len(money), ","))
                break
            if _kit_is_number(money[k]):
                parts.append(_kit_dollars(money[k]))
            else:
                parts.append(_kit_short(money[k]))
        print(label, ", ".join(parts) if parts else "none")
    elif _kit_is_number(money):
        print(label, _kit_dollars(money))
    else:
        print(label, _kit_short(money))


def show_value(label, value):
    """Print a value after a label: show_value("nabla =", 174). Silent for None (a line above says why)."""
    if value is None:
        return
    print(label, _kit_short(value))


def _kit_money(x):
    """-$172.24, $73.57"""
    text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def _kit_axis(x):
    """-$150, $250"""
    text = "$%d" % abs(int(round(x)))
    return "-" + text if x < 0 else text


def plot_learning_curve(points, best=_KIT_BEST_LINE, label="optimal policy"):
    """The learning curve of page 3, as text: across, the number of updates; up, the average return after them. o is a
    point (one column for every few updates when there are many), and the row of dashes marks the optimal policy's
    average return on the luck of the last update. Silent for None (a line above says why)."""
    if points is None:
        return
    ok = isinstance(points, list) and len(points) >= 2
    if ok:
        for x in points:
            if not _kit_is_number(x):
                ok = False
    if not ok:
        print("⏳ No learning curve yet: plot_learning_curve needs a list of at least two averages. It got %s."
              % _kit_short(points))
        return
    n = len(points) - 1                      # the updates
    columns = min(n + 1, 61)
    where = []                               # the update that each column shows
    for c in range(columns):
        where.append(int(round(c * n / (columns - 1))))
    values = []
    for i in where:
        values.append(points[i])
    low, high = min(values + [best]), max(values + [best])
    step = 10
    for size in (10, 25, 50, 100, 250, 500, 1000):
        step = size
        if (high - low) / size <= 18:
            break
    top = _kit_math.ceil(high / step) * step
    bottom = _kit_math.floor(low / step) * step
    rows = int(round((top - bottom) / step)) + 1
    grid = []
    for r in range(rows):
        grid.append([" "] * columns)
    best_row = int(round((top - best) / step))
    for c in range(columns):
        grid[best_row][c] = "-"
    for c in range(columns):
        r = int(round((top - values[c]) / step))
        grid[r][c] = "o"
    every = max(1, int(round(n / (columns - 1)))) if columns > 1 else 1
    print("The learning curve: the average return after each update" +
          (" (one o for every %d updates)" % every if every > 1 else ""))
    print()
    for r in range(rows):
        line = "%8s |" % _kit_axis(top - r * step) + "".join(grid[r])
        if r == best_row:
            line += "  %s, %s" % (label, _kit_money(best))
        print(line.rstrip())
    print(" " * 9 + "+" + "-" * columns)
    ticks = 50 if n >= 200 else (10 if n >= 40 else 1)
    marks = [" "] * (columns + 6)
    for t in range(0, n + 1, ticks):
        c = int(round(t * (columns - 1) / n)) if n else 0
        text = str(t)
        for j in range(len(text)):
            if c + j < len(marks):
                marks[c + j] = text[j]
    print(" " * 10 + "".join(marks).rstrip() + "   updates")
    print("first point %s, last point %s" % (_kit_money(points[0]), _kit_money(points[-1])))


print("✅ Old Bessie's kit is ready.")

In [ ]:
#@title The functions of part 1, given (run me too) { display-mode: "form" }
# The functions of part 1, given, and my_table, its made-up probability table. You do not need to read this cell.


my_table = [[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]


def make_roulette(probabilities):
    # probabilities: a row of probabilities, one per action.
    # Gives the ends of the slots: a running total of the probabilities, such as [0.6, 0.9, 1.0].
    ends = []
    running = 0                         # where the slots so far end: nowhere yet
    for probability in probabilities:
        running = running + probability  # this slot ends one probability further
        ends.append(running)             # remember where this slot ends
    return ends


def where_it_lands(ends, ball):
    # ends: the ends of the slots, from make_roulette. ball: a number between 0 and 1.
    # Gives the number of the first slot whose end is above the ball: 0 for RUN, 1 for SERVICE, 2 for REPLACE.
    for slot in range(len(ends)):
        if ball < ends[slot]:           # the ball is below the end of this slot
            return slot
    return len(ends) - 1                # a ball at the very end of the wheel: the last slot


def pick_action(policy, state):
    # policy: a probability table, one row per state. state: the van's state, the number of a row.
    # Gives an action sampled with the probabilities of that row: 0 for RUN, 1 for SERVICE, 2 for REPLACE.
    probabilities = policy[state]        # the row of the state
    ends = make_roulette(probabilities)  # its roulette
    ball = spin()                        # the ball
    return where_it_lands(ends, ball)    # where the ball lands


def five_weeks(policy):
    # policy: a probability table. Plays five weeks of the van from HEALTHY, one after the other.
    # Gives the money of the five weeks, in a list.
    state = HEALTHY
    action = pick_action(policy, state)             # week 1: the policy picks an action in the van's state
    state, money_1 = transition(state, action)      # week 1: the van's week; state is now next week's state
    action = pick_action(policy, state)             # week 2, the policy's action
    state, money_2 = transition(state, action)      # week 2, the van's week
    action = pick_action(policy, state)             # week 3, the policy's action
    state, money_3 = transition(state, action)      # week 3, the van's week
    action = pick_action(policy, state)             # week 4, the policy's action
    state, money_4 = transition(state, action)      # week 4, the van's week
    action = pick_action(policy, state)             # week 5, the policy's action
    state, money_5 = transition(state, action)      # week 5, the van's week
    return [money_1, money_2, money_3, money_4, money_5]


def total_of_five(money):
    # money: the money of five weeks, in a list. Gives their total.
    total = money[0]                    # week 1
    total = total + money[1]            # week 2 added
    total = total + money[2]            # week 3 added
    total = total + money[3]            # week 4 added
    total = total + money[4]            # week 5 added
    return total


def play_episode(policy, weeks):
    # policy: a probability table. weeks: how many weeks the episode lasts.
    # Plays an episode of the van from HEALTHY. Gives its rewards: the money of every week, in a list.
    rewards = []
    state = HEALTHY
    for week in range(weeks):
        action = pick_action(policy, state)         # the policy picks an action in the van's state
        state, money = transition(state, action)    # the van's week
        rewards.append(money)                       # remember the week's money
    return rewards


def episode_return(rewards):
    # rewards: the money of every week of an episode, in a list.
    # Gives the episode's return G1: week 1's money counts in full, and each later week's money counts
    # GAMMA = 0.9 times as much as the week before.
    G1 = 0
    weight = 1                          # week 1 counts in full
    for money in rewards:
        G1 = G1 + weight * money        # add the week's money, times its weight
        weight = weight * GAMMA         # the next week counts GAMMA times as much
    return G1


print("✅ The functions of part 1 are ready: pick_action, play_episode, episode_return and my_table.")

In [ ]:
#@title The checks of part 2 (run me too) { display-mode: "form" }
# The checks of notebook 1 and play. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N1_TESTS = {'play_episodes': [{'about': 'my_table, 4 episodes of 3 weeks, on luck 1 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 3, 4], 'kind': 'table', 'want': [[95, 95, -240], [-50, 95, 72], [95, -50, 95], [-130, 95, 72]], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table, 2 episodes of 13 weeks, on luck 2 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 13, 2], 'kind': 'table', 'want': [[-50, 95, 72, -130, -130, -50, -50, 95, -50, -50, -185, -130, 95], [95, -50, -130, 95, -50, -50, -50, 95, -50, -50, 95, -50, -130]], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [5]]]}, {'about': 'a policy that always runs the van, [1, 0, 0] in every row, 3 episodes of 2 weeks, on luck 3 of the check', 'args': [[[1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0]], 2, 3], 'kind': 'table', 'want': [[95, 95], [95, 72], [95, 72]], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [7]]]}, {'about': 'my_table, 0 episodes of 5 weeks, on luck 4 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 5, 0], 'kind': 'table', 'want': [], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [1]]]}], 'returns': [{'about': 'the episodes [[95, 72], [-185, 95]]', 'args': [[[95, 72], [-185, 95]]], 'kind': 'numbers', 'want': [159.8, -99.5], 'tol': 1e-09, 'keep': [0]}, {'about': "the episodes [page 2's episode]", 'args': [[[95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]]], 'kind': 'numbers', 'want': [154.39614434222], 'tol': 1e-09, 'keep': [0]}, {'about': 'the episodes []', 'args': [[]], 'kind': 'numbers', 'want': [], 'tol': 1e-09, 'keep': [0]}], 'average': [{'about': 'the numbers [1, 2, 3, 4]', 'args': [[1, 2, 3, 4]], 'kind': 'number', 'want': 2.5, 'tol': 1e-09, 'keep': [0]}, {'about': 'the numbers [-172.24]', 'args': [[-172.24]], 'kind': 'number', 'want': -172.24, 'tol': 1e-09, 'keep': [0]}, {'about': 'the numbers [10, -10, 3]', 'args': [[10, -10, 3]], 'kind': 'number', 'want': 1.0, 'tol': 1e-09, 'keep': [0]}, {'about': 'the numbers [154.3961, 0]', 'args': [[154.39614434222, 0]], 'kind': 'number', 'want': 77.19807217111, 'tol': 1e-09, 'keep': [0]}], 'average_return': [{'about': "page 3's preferences, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -172.24, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's preferences with the preference for RUN in HEALTHY one hop higher, on page 3's luck", 'args': [[[0.41000000000000003, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -170.5, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "the preferences of warm-up 1, on page 3's luck", 'args': [[[0.4, -0.6, 5], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -945.97, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's preferences, on luck 2 of the check", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -156.92, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [2001]]]}]}
_N1_SLIPS = {'play_episodes': [{'answers': [[[95, 95, -240, -130], [95, 72, -50, -130], [95, -130, 95, 72], [95, 95, 95, 95]], [[-50, 95], [95, -50]], [[95, 95, 72], [95, 95, 72], [95, 72, 40]], []], 'hint': 'each episode lasts weeks weeks: play_episode(policy, weeks)'}, {'answers': [[-13.900000000000006, 93.82000000000001, 126.95, 13.820000000000007], [-216.2658123945051, -89.68464718753006], [180.5, 159.8, 159.8], []], 'hint': 'keep the whole episode, its list of rewards: episodes.append(episode)'}], 'returns': [{'answers': [[[95, 72], [-185, 95]], [[95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]], []], 'hint': "append the episode's G1, episode_return(episode), and not the episode itself"}], 'average': [{'answers': [10, -172.24, 3, 154.39614434222], 'hint': 'divide the total by how many numbers there are: total / len(numbers)'}, {'answers': [1.0, -172.24, 1.0, 0.0], 'hint': 'add x to the total so far: total = total + x'}], 'average_return': [{'answers': [-439.56, -429.19, -439.56, -426.18], 'hint': 'turn the preferences into probabilities first: policy = probability_table(preferences)'}, {'answers': [-160.86, -159.11, -932.64, -131.85], 'hint': 'page 3 averages EPISODES = 1,000 episodes: play_episodes(policy, WEEKS, EPISODES)'}, {'answers': [-215.06, -215.06, -1268.47, -226.42], 'hint': 'play_episodes takes the weeks of an episode first, then how many episodes: play_episodes(policy, WEEKS, EPISODES)'}]}
_N1_SUCCESS = {'play_episodes': 'play_episodes works: the episodes come back in a list, one list of rewards per episode.', 'returns': 'returns works: one G1 per episode, in the order of the episodes.', 'average': 'average works: the average of [1, 2, 3, 4] is 2.5.', 'average_return': "average_return works: page 3's preferences give -$172.24 on page 3's luck, the first point of page 3's learning curve."}
_N1_DEFAULT = {'play_episodes': 'episode = play_episode(policy, weeks), then episodes.append(episode)', 'returns': 'G1s.append(episode_return(episode))', 'average': 'total = total + x in the loop, then return total / len(numbers)', 'average_return': 'policy = probability_table(preferences), then episodes = play_episodes(policy, WEEKS, EPISODES), then G1s = returns(episodes)'}
_N1_KEEP_HINT = {'play_episodes': 'play_episodes reads the probability table and leaves it as it is', 'returns': 'returns reads the episodes and leaves them as they are', 'average': 'average reads the numbers and leaves them as they are', 'average_return': 'average_return reads the table of preferences and leaves it as it is: probability_table gives a new table'}
_N1_TASKS = {'play_episodes': 'task A', 'returns': 'task B', 'average': 'task C', 'average_return': 'task D'}
_N1_NEEDS = {'play_episodes': ['play_episode'], 'returns': ['episode_return'], 'average': [], 'average_return': ['play_episodes', 'returns', 'average']}
_N1_LONG = []
_N1_NONE_HINT = {}
_N1_DICE_OF = 'spin'
_N1_GAPS = {}
_N1_WORKING = {}
_N1_DICE = {}


def _n1_num(x):
    """A number with at most 4 decimals: 154.3961, 2.14, 174."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x != x or x in (float("inf"), float("-inf")):
        return repr(x)
    if isinstance(x, int):
        return str(x)
    text = ("%.4f" % x).rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    if text == "0" and x != 0:
        text = "%.2g" % x
    return text


def _n1_text(value):
    """Numbers with at most 4 decimals, lists of them in brackets."""
    if isinstance(value, (list, tuple)):
        parts = []
        for x in value:
            parts.append(_n1_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n1_num(value)


def _n1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = _n1_text(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 72:
        text = text[:69] + "..."
    return text


def _n1_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _n1_same(kind, got, want, tol):
    """True when got is the right answer want, within tol."""
    try:
        if kind == "number":
            return _n1_is_number(got) and abs(got - want) <= tol
        if kind == "action":
            return _n1_is_number(got) and got == want
        if kind in ("numbers", "table"):
            if not isinstance(got, (list, tuple)) or len(got) != len(want):
                return False
            for k in range(len(want)):
                if not _n1_same("numbers" if kind == "table" else "number", got[k], want[k], tol):
                    return False
            return True
    except Exception:
        return False
    return False


def _n1_like(got, answer, tol):
    """True when got has the shape and the numbers of a slip's answer, within tol: a slip may give a number where a
    list is due (return total), or a list of numbers where a list of lists is due."""
    if isinstance(answer, (list, tuple)):
        if not isinstance(got, (list, tuple)) or len(got) != len(answer):
            return False
        for k in range(len(answer)):
            if not _n1_like(got[k], answer[k], tol):
                return False
        return True
    return _n1_is_number(got) and _n1_is_number(answer) and abs(got - answer) <= tol


def _n1_copy(value):
    """A deep copy of lists of lists of numbers."""
    if isinstance(value, list):
        out = []
        for x in value:
            out.append(_n1_copy(x))
        return out
    return value


def _n1_dice(seed, first, count):
    """The dice of count episodes of a seed from episode first, rolled by the kit once per seed and kept."""
    rolled = _N1_DICE.get(seed)
    if rolled is None or len(rolled) < first + count:
        if len(_N1_DICE) > 3:
            _N1_DICE.clear()
        rolled = globals()[_N1_DICE_OF](seed, max(first + count, 10))
        _N1_DICE[seed] = rolled
    return rolled[first:first + count]


def _n1_args(test):
    """Fresh inputs for a test: copies of the lists, and the dice it names."""
    args = []
    for a in test["args"]:
        if isinstance(a, dict) and "dice" in a:
            args.append(_n1_dice(a["dice"][0], a["dice"][1], a["dice"][2]))
        elif isinstance(a, dict) and "episode" in a:
            args.append(_n1_dice(a["episode"][0], a["episode"][1], 1)[0])
        else:
            args.append(_n1_copy(a))
    return args


def _n1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N1_GAPS:
        return _N1_GAPS[code]
    # A gap such as "total = ..." stays in the compiled function; a gap such as "if ...:" only in the source, which
    # Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N1_GAPS[code] = gap
    return gap


def _n1_holds_gap(value):
    """True when a value, or a list inside it, still holds a ... (a gap of a cell above)."""
    if value is Ellipsis:
        return True
    if isinstance(value, (list, tuple)):
        for x in value:
            if _n1_holds_gap(x):
                return True
    return False


def _n1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n1_key(name, seen=()):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    if name in seen:
        return ()
    fn = globals().get(name)
    key = [getattr(fn, "__code__", None)]
    for other in _N1_NEEDS.get(name, []):
        key.extend(_n1_key(other, seen + (name,)))
    return tuple(key)


def _n1_call(fn, args):
    """Run fn with its prints silenced: {"answer", "error" (None when the call went through), "line" (the line of
    your code where it happened)}. Never raises."""
    import contextlib, io, traceback
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*args)
    except Exception as error:
        run["error"] = "%s: %s" % (type(error).__name__, error)
        try:
            skip = (_n1_call.__code__.co_filename, globals()[_N1_DICE_OF].__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    return run


def _n1_explain(run):
    """A hint for an error."""
    text = run["error"]
    if "NoneType" in text:
        return 'a value in that line is None: a function it calls gave nothing back (is that function finished, with its return line?), or a name holds None'
    if "ellipsis" in text:
        return "a ... is still in a function that this one calls: finish that function first"
    if "list indices must be integers" in text:
        return 'a place in [ ] must be a whole number: a state or an action, such as policy[state] for the row of a state and row[action] for one of its numbers'
    if "is not subscriptable" in text:
        return "that line uses [ ] on a single number: only a list has places to look up"
    if "is not iterable" in text:
        return ("a single number stands where a list was expected: check the names on the left of each = "
                "(update_preference changes the table itself and gives back only the new preference)")
    if 'transition needs' in text:
        return 'the van got a state or an action it does not know: check what pick_action gives back'
    if "ZeroDivisionError" in text:
        return "a division by zero: divide by a number that cannot be 0, such as the total or len(...)"
    if "NameError" in text:
        return "a name in that line is not defined: check its spelling, and run the cells above it first"
    if "can only concatenate" in text or "unsupported operand" in text:
        return "that line mixes a list and a number: take one number of the list with [ ] first"
    if "positional argument" in text:
        return "a function got more or fewer inputs than it takes: compare the call with the function's def line"
    if "out of range" in text:
        return "a place in [ ] is past the end of the list: count from 0, the last place is len(...) - 1"
    return "read that line again, compare it with the hints, run the function's cell, then this check again"


def _n1_failure(name, fn):
    """None when fn passes every test of its task, else (head, what, hint). The kit's state (its luck) is put back
    afterwards when the kit can save it."""
    save = globals().get("_kit_save")
    saved = save() if callable(save) else None
    try:
        return _n1_failure_of(name, fn)
    finally:
        if saved is not None:
            globals()["_kit_restore"](saved)


def _n1_failure_of(name, fn):
    tests = _N1_TESTS[name]
    for k in range(len(tests)):
        test = tests[k]
        args = _n1_args(test)
        kept = _n1_copy(args)
        for setup in test.get("setup", []):
            globals()[setup[0]](*setup[1])
        run = _n1_call(fn, args)
        head = "On " + test["about"]
        if run["error"] is not None:
            where = ""
            if run["line"]:
                where = ", in the line  %s " % run["line"]
            return (head, "%s stopped with an error, %s%s" % (name, run["error"], where), _n1_explain(run))
        got = run["answer"]
        if not _n1_same(test["kind"], got, test["want"], test["tol"]):
            if got is None:
                return (head, "%s gave None" % name, _N1_NONE_HINT.get(
                    name, "a function gives its answer with return: keep the return line at its end"))
            hint = _N1_DEFAULT[name]
            for slip in _N1_SLIPS[name]:
                answer = slip["answers"][k]
                if answer is not None and _n1_like(got, answer, test["tol"]):
                    hint = slip["hint"]
                    break
            return (head, "%s should give %s, and yours gives %s" % (name, _n1_short(test["want"]),
                                                                    _n1_short(got)), hint)
        for i in test.get("keep", []):
            if args[i] != kept[i]:
                return (head, "%s changed its input %s" % (name, _n1_short(kept[i])), _N1_KEEP_HINT[name])
        if test.get("table"):
            t = test["table"]
            if not _n1_same("table", args[t["arg"]], t["want"], test["tol"]):
                return (head, "after %s the table of preferences should be %s, and it is %s" % (
                    name, _n1_short(t["want"]), _n1_short(args[t["arg"]])), _N1_KEEP_HINT[name])
    return None


def _n1_works(name):
    """True when the function called name exists, is finished, the functions it uses work, and it passes its tests.
    Remembered for each version of it."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n1_has_gap(fn):
        return False
    for other in _N1_NEEDS.get(name, []):
        if _n1_uses(fn, other) and not _n1_works(other):
            return False
    if name not in _N1_TESTS:
        return True
    key = _n1_key(name)
    if key not in _N1_WORKING:
        _N1_WORKING[key] = _n1_failure(name, fn) is None
    return _N1_WORKING[key]


def _n1_waits(name, fn, cell):
    """One calm line and True when fn cannot run yet: it is not finished, or a function it uses does not work yet."""
    task = ""
    if name in _N1_TASKS:
        task = " (%s)" % _N1_TASKS[name]
    if _n1_has_gap(fn):
        print("⏳ %s is not finished yet%s: replace each ... with your code, run its cell, then %s again."
              % (name, task, cell))
        return True
    for other in _N1_NEEDS.get(name, []):
        if not _n1_uses(fn, other) or _n1_works(other):
            continue
        if other in _N1_TASKS:
            needed = globals().get(other)
            what = "does not work yet"
            if callable(needed) and _n1_has_gap(needed):
                what = "is not finished yet"
            print("⏳ %s uses your %s (%s), which %s. Get check_%s(%s) to ✅ first, then run %s again."
                  % (name, other, _N1_TASKS[other], what, other, other, cell))
            return True
        # a given function: it waits for the tasks it uses
        for deeper in _N1_NEEDS.get(other, []):
            if deeper in _N1_TASKS and not _n1_works(deeper):
                print("⏳ %s uses %s, which uses your %s (%s). Get check_%s(%s) to ✅ first, then run %s again."
                      % (name, other, deeper, _N1_TASKS[deeper], deeper, deeper, cell))
                return True
        print("⏳ %s uses %s, which does not work yet: run its cell again, then %s." % (name, other, cell))
        return True
    return False


def _n1_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n1_waits(name, fn, "this check"):
        return
    failure = _n1_failure(name, fn)
    if failure is None:
        print("✅ " + _N1_SUCCESS[name])
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint + ".")


def _n1_safe_check(name, fn):
    try:
        _n1_check(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_play_episodes(fn):
    """Task a: several episodes at once, each on luck the check deals"""
    _n1_safe_check("play_episodes", fn)

def check_returns(fn):
    """Task b: the G1 of every episode of several lists of episodes"""
    _n1_safe_check("returns", fn)

def check_average(fn):
    """Task c: the average of several lists of numbers"""
    _n1_safe_check("average", fn)

def check_average_return(fn):
    """Task d: the average return of several tables of preferences, each on luck the check deals"""
    _n1_safe_check("average_return", fn)


def play(fn, *inputs):
    """Runs one of your functions and gives its answer: play(average_return, preferences) is
    average_return(preferences). When the function is not finished yet, uses a function that does not work yet, gets
    None from a cell above or stops with an error, play prints one calm line and gives None, so the cells below
    still run."""
    import time
    try:
        if not callable(fn):
            print("⏳ play needs your function itself, as in play(average_return, preferences).")
            return None
        name = getattr(fn, "__name__", "your function")
        if _n1_waits(name, fn, "this cell"):
            return None
        for x in inputs:
            if x is None:
                print("⏳ %s waits for a value that a cell above did not give yet (its ⏳ line says why)." % name)
                return None
            if _n1_holds_gap(x):
                print("⏳ %s waits for a value that still holds a ...: fill it in above, run its cell, then this cell."
                      % name)
                return None
        if name in _N1_LONG and not _n1_works(name):
            print("⏳ %s waits until it passes its check: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return None
        long = name in _N1_LONG
        if long and callable(globals().get("_kit_before_long")):
            globals()["_kit_before_long"]()
        start = time.time()
        try:
            answer = fn(*inputs)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check says more." % (name, type(error).__name__, error))
            return None
        finally:
            if long and callable(globals().get("_kit_after_long")):
                globals()["_kit_after_long"]()
        seconds = time.time() - start
        if seconds > 5:
            print("⏱️ %s took %d seconds." % (name, round(seconds)))
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the folded cells at the top again, then this cell." % error)
        return None

_N1_TABLE = [[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]
_N1_START_PREFERENCES = [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]


def _n1_ready(name, cell):
    """True when your function called name exists and passes its check; else one calm line."""
    fn = globals().get(name)
    if not callable(fn):
        print("⏳ %s is not there yet: run the cell of %s first, then %s." % (name, _N1_TASKS[name], cell))
        return False
    if _n1_waits(name, fn, cell):
        return False
    if not _n1_works(name):
        print("⏳ This cell waits until %s passes its check: get check_%s(%s) to ✅ first, then run %s again."
              % (name, name, name, cell))
        return False
    return True


def _n1_probabilities_ready(probabilities, cell):
    """True when probabilities is a row of three numbers; else one calm line."""
    if probabilities is None or _n1_holds_gap(probabilities):
        print("⏳ %s waits for a row of probabilities: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    if not isinstance(probabilities, list) or len(probabilities) != 3:
        print("⏳ %s needs a row of three probabilities, such as my_table[HEALTHY]." % cell)
        return False
    return True


def _n1_policy_ready(policy, cell):
    """True when policy is a probability table without gaps; else one calm line."""
    if policy is None or _n1_holds_gap(policy):
        print("⏳ %s waits for a probability table: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    return True


def check_table(table):
    """Warm-up 1: my_table, the made-up probability table."""
    try:
        if _n1_holds_gap(table):
            print("⏳ my_table is not finished yet (warm-up 1): replace each ... with a row of three probabilities, run "
                  "its cell, then this check again.")
            return
        if not isinstance(table, list) or len(table) != 4:
            print("❌ my_table should be a list of four rows, one per state, and yours is %s." % _n1_short(table))
            print("   💡 Hint: write one row per line, each in [ ] and followed by a comma, all inside the outer [ ].")
            return
        for s in range(4):
            row = table[s]
            if not _n1_same("numbers", row, _N1_TABLE[s], 1e-9):
                print("❌ %s's row should be %s, and yours is %s." % (STATES[s], _n1_text(_N1_TABLE[s]),
                                                                   _n1_short(row)))
                hint = "copy the row from the table above, three numbers in the order RUN, SERVICE, REPLACE"
                for other in range(4):
                    if other != s and _n1_same("numbers", row, _N1_TABLE[other], 1e-9):
                        hint = ("that is %s's row: the rows go in the order HEALTHY, WORN, SHAKY, FAILING"
                                % STATES[other])
                if isinstance(row, list) and len(row) == 3 and _n1_same("numbers", sorted(row), sorted(_N1_TABLE[s]),
                                                                        1e-9) and hint.startswith("copy"):
                    hint = "the numbers of a row go in the order of the columns: RUN, SERVICE, REPLACE"
                print("   💡 Hint: " + hint + ".")
                return
        print("✅ my_table is the made-up probability table: four rows, one per state, and each row adds up to 1.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_reading(probability, row):
    """Warm-up 2: a number and a row read from my_table."""
    try:
        if probability is Ellipsis or _n1_holds_gap(row):
            print("⏳ Warm-up 2 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        want_probability = _N1_TABLE[FAILING][REPLACE]
        want_row = _N1_TABLE[SHAKY]
        if not _n1_same("number", probability, want_probability, 1e-9):
            print("❌ probability should be %s, the probability of REPLACE when the van is FAILING, and yours is %s."
                  % (_n1_text(want_probability), _n1_short(probability)))
            hint = "the row comes first, then the column: my_table[FAILING][REPLACE]"
            if _n1_same("number", probability, _N1_TABLE[FAILING][SERVICE], 1e-9) or _n1_same(
                    "number", probability, _N1_TABLE[FAILING][RUN], 1e-9):
                hint = "that is another column of FAILING's row: REPLACE is the column REPLACE, number 2"
            print("   💡 Hint: " + hint + ".")
            return
        if not _n1_same("numbers", row, want_row, 1e-9):
            print("❌ row should be SHAKY's row, %s, and yours is %s." % (_n1_text(want_row), _n1_short(row)))
            print("   💡 Hint: a row is the table with one [ ]: my_table[SHAKY].")
            return
        print("✅ Both right: my_table[FAILING][REPLACE] is %s, and my_table[SHAKY] is %s." % (
            _n1_text(want_probability), _n1_text(want_row)))
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_garage(next_state, money):
    """Warm-up 3: one week of a WORN van in the garage, transition(WORN, SERVICE)."""
    try:
        if next_state is None and money is None:
            print("⏳ Warm-up 3 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        for running, state, cash in _KIT_WEEK[WORN][SERVICE]:
            if next_state == state and money == cash:
                print("✅ A week of a WORN van in the garage: next week it is %s, and the week's money is %s." % (
                    STATES[next_state], _kit_dollars(money)))
                return
        print("❌ A WORN van in the garage costs %s for the week and comes back HEALTHY or WORN; your week gave the "
              "state %s and the money %s." % (_kit_dollars(_KIT_SHOP), _n1_short(next_state), _n1_short(money)))
        hint = "call transition with the state WORN first, then the action SERVICE: transition(WORN, SERVICE)"
        if money == -_KIT_NEW_VAN:
            hint = "that is the money of REPLACE: the action of the garage is SERVICE"
        elif _n1_is_number(money) and money != -_KIT_SHOP:
            hint = "that money comes from the route, the action RUN: the action of the garage is SERVICE"
        print("   💡 Hint: " + hint + ".")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def _n1_softmax(row):
    largest = max(row)
    powers = []
    for x in row:
        powers.append(exp(x - largest))
    total = 0
    for x in powers:
        total = total + x
    out = []
    for x in powers:
        out.append(x / total)
    return out


def check_my_preferences(preferences):
    """Warm-up 1: page 3's preferences with the preference for REPLACE in HEALTHY set to 5."""
    try:
        if preferences is None or _n1_holds_gap(preferences):
            print("⏳ Warm-up 1 is not finished yet: replace the ... with your code, run its cell, then this check "
                  "again.")
            return
        if not _kit_is_table(preferences):
            print("❌ my_preferences should be a table of preferences, four rows of three numbers, and it is %s."
                  % _n1_short(preferences))
            print("   💡 Hint: start from my_preferences = start_preferences(), then change one preference.")
            return
        changed = []
        for s in range(4):
            for a in range(3):
                if preferences[s][a] != _N1_START_PREFERENCES[s][a]:
                    changed.append([s, a])
        if len(changed) == 0:
            print("⏳ my_preferences is still page 3's table: set the preference for REPLACE in HEALTHY to 5 in the "
                  "cell of warm-up 1, run it, then this check again.")
            return
        if changed == [[HEALTHY, REPLACE]] and preferences[HEALTHY][REPLACE] == 5:
            print("✅ The preference for REPLACE in HEALTHY is 5: HEALTHY's probability of REPLACE went from %.2f to "
                  "%.2f." % (
                _n1_softmax(_N1_START_PREFERENCES[HEALTHY])[REPLACE], _n1_softmax(preferences[HEALTHY])[REPLACE]))
            return
        if len(changed) > 1:
            print("❌ %d preferences of my_preferences differ from page 3's; warm-up 1 changes one." % len(changed))
            print("   💡 Hint: start again from my_preferences = start_preferences(), then change only "
                  "my_preferences[HEALTHY][REPLACE].")
            return
        s, a = changed[0]
        if [s, a] != [HEALTHY, REPLACE]:
            print("❌ You changed the preference for %s in %s; warm-up 1 changes the preference for REPLACE in "
                  "HEALTHY." % (
                ACTIONS[a], STATES[s]))
            print("   💡 Hint: the row comes first, then the column: my_preferences[HEALTHY][REPLACE] = 5.")
            return
        print("❌ The preference for REPLACE in HEALTHY is %s; warm-up 1 sets it to 5." % _n1_short(preferences[s][a]))
        print("   💡 Hint: my_preferences[HEALTHY][REPLACE] = 5.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def throw_balls(probabilities, throws):
    """Throws the ball throws times on the roulette of a row of probabilities, with make_roulette and
    where_it_lands, and prints how often each action came."""
    try:
        if not _n1_probabilities_ready(probabilities, "throw_balls"):
            return
        if not _n1_ready("make_roulette", "this cell") or not _n1_ready("where_it_lands", "this cell"):
            return
        ends = make_roulette(probabilities)
        counts = [0, 0, 0]
        for k in range(throws):
            slot = where_it_lands(ends, spin())
            counts[slot] = counts[slot] + 1
        print("In %s throws:   RUN %d times, SERVICE %d times, REPLACE %d times" % (
            format(throws, ","), counts[0], counts[1], counts[2]))
        print("Their shares:      RUN %.2f, SERVICE %.2f, REPLACE %.2f" % (
            counts[0] / throws, counts[1] / throws, counts[2] / throws))
        print("The probabilities: RUN %.2f, SERVICE %.2f, REPLACE %.2f" % (
            probabilities[0], probabilities[1], probabilities[2]))
    except Exception as error:
        print("❌ throw_balls could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_picks(policy, picks):
    """Your pick_action, picks times in each state of a probability table: how often each action came, and the
    shares."""
    try:
        if not _n1_policy_ready(policy, "show_picks"):
            return
        if not _n1_ready("pick_action", "this cell"):
            return
        counts = []
        shares = []
        for s in range(4):
            counts.append([0, 0, 0])
            for k in range(picks):
                action = pick_action(policy, s)
                counts[s][action] = counts[s][action] + 1
            shares.append([counts[s][0] / picks, counts[s][1] / picks, counts[s][2] / picks])
        print("Your pick_action, %s times in each state: how often each action came" % format(picks, ","))
        show_counts(counts)
        print()
        print("Their shares, to set beside the probability table")
        show_probabilities(shares)
    except Exception as error:
        print("❌ show_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def count_picks(how_many):
    """Your pick_preference, how_many times: how often each preference came."""
    try:
        if not _n1_ready("pick_preference", "this cell"):
            return
        counts = [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
        for k in range(how_many):
            state, action = pick_preference()
            counts[state][action] = counts[state][action] + 1
        print("Your pick_preference, %s times: how often each preference came" % format(how_many, ","))
        show_counts(counts)
    except Exception as error:
        print("❌ count_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_episode(policy, weeks):
    """Plays one episode with your pick_action and the kit's transition, and prints it week by week."""
    try:
        if not _n1_policy_ready(policy, "show_episode"):
            return
        if not _n1_ready("pick_action", "this cell"):
            return
        state = HEALTHY
        print("week  state     action     money")
        for week in range(weeks):
            action = pick_action(policy, state)
            next_state, money = transition(state, action)
            print("%4d  %-8s  %-8s  %6s" % (week + 1, STATES[state], ACTIONS[action], _kit_dollars(money)))
            state = next_state
    except Exception as error:
        print("❌ show_episode could not run (%s). Run the folded cells at the top again, then this cell." % error)

print("✅ The checks of part 2 are ready.")

## 1. Many episodes and their average

One episode says little: luck decides much of its $G_1$. Page 3 judged a policy by the average return of `EPISODES`
= 1,000 episodes. Three small functions get there: the episodes, their $G_1$, their average.

### 🎯 Task A. Many episodes

Write `play_episodes(policy, weeks, how_many)`: `how_many` episodes of the same policy, collected in a list.

In [ ]:
# ✅ SOLUTION, task A: many episodes
def play_episodes(policy, weeks, how_many):
    # Plays how_many episodes of weeks weeks with the same policy.
    # Gives the episodes in a list, each episode the list of its rewards.
    episodes = []
    for k in range(how_many):
        episode = play_episode(policy, weeks)  # one episode
        episodes.append(episode)               # remember the episode
    return episodes

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# One episode is play_episode(policy, weeks): the same policy and the same number of weeks every time.
# episodes.append(...) puts the episode, its whole list of rewards, at the end of episodes.

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
#     episode = play_episode(policy, weeks)
#     episodes.append(episode)

In [ ]:
# Check task A
check_play_episodes(play_episodes)

### 🎯 Task B. The $G_1$ of every episode

Write `returns(episodes)`: the return of each episode, in a list.

In [ ]:
# ✅ SOLUTION, task B: the G1 of every episode
def returns(episodes):
    # episodes: a list of episodes, each the list of its rewards.
    # Gives the return G1 of every episode, in a list.
    G1s = []
    for episode in episodes:
        G1s.append(episode_return(episode))     # the episode's G1
    return G1s

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# episode_return(episode) is the G1 of one episode, a single number. Append it to G1s.

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
#     G1s.append(episode_return(episode))

In [ ]:
# Check task B
check_returns(returns)

### 🎯 Task C. The average

Write `average(numbers)`: the total of the numbers divided by how many there are, `len(numbers)`.

In [ ]:
# ✅ SOLUTION, task C: the average
def average(numbers):
    # numbers: a list of numbers. Gives their average: their total divided by how many there are.
    total = 0
    for x in numbers:
        total = total + x               # add x to the total
    return total / len(numbers)         # the total divided by how many numbers there are

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# The total grows by x in every round of the loop. After the loop, divide it by len(numbers), how many numbers
# there are.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
#     total = total + x
#     return total / len(numbers)

In [ ]:
# Check task C
check_average(average)

In [ ]:
# 1,000 episodes of my_table's policy and their average return: run it a few times
episodes = play(play_episodes, my_table, WEEKS, EPISODES)
G1s = play(returns, episodes)
show_money("Their G1s:", G1s)
show_money("The average return:", play(average, G1s))

**Reading the result.** Each time the cell runs, 1,000 new episodes give a new average: ten sets of 1,000 episodes of
`my_table` gave averages from -\$12.20 to \$12.52. To compare two tables of preferences, page 3 played both on the same
episodes; section 2 does the same.

## 2. The table of preferences

proto-REINFORCE does not change probabilities directly. It keeps a table of preferences, one number for each state
and action, and softmax turns each row of preferences into a row of probabilities: the larger a preference, the
larger its probability. Page 3 started from preferences picked at random between -10 and 10; `start_preferences()`
gives a fresh copy of them.

In [ ]:
# Page 3's table of preferences
preferences = start_preferences()
show_preferences(preferences)

### Softmax

The folded cell below holds softmax, given: `softmax(row_of_preferences)` turns a row of preferences into a row of
probabilities, and `probability_table(preferences)` does it for every row. Run it. Reading its code is optional, for
illustration only.

In [ ]:
#@title Softmax, given: optional to read (run me) { display-mode: "form" }
# Softmax, given. Reading this cell is optional, for illustration only.

def softmax(row_of_preferences):
    # A row of preferences to a row of probabilities: e to the power of each preference, divided by their total, so that
    # the probabilities add up to 1 and a larger preference gets a larger probability. The largest preference is taken
    # away first, as on page 3: the probabilities stay the same, and exp never meets a number too large for it.
    largest = max(row_of_preferences)
    powers = []
    for preference in row_of_preferences:
        powers.append(exp(preference - largest))
    total = 0
    for power in powers:
        total = total + power
    probabilities = []
    for power in powers:
        probabilities.append(power / total)
    return probabilities


def probability_table(preferences):
    # A table of preferences to its probability table: softmax on every row.
    table = []
    for row in preferences:
        table.append(softmax(row))
    return table


print("✅ softmax and probability_table are ready.")

In [ ]:
# The probability table of page 3's preferences
show_probabilities(probability_table(preferences))

**Reading the result.** SHAKY's preference for REPLACE, 10, is far above its other two preferences, so SHAKY's
probability of REPLACE is almost 1. WORN's preferences for RUN and SERVICE, -0.7 and -0.8, are close, and so are their
probabilities.

### 🎯 Warm-up 1. A preference changed by hand

In a copy of page 3's preferences, set the preference for REPLACE in HEALTHY to 5, and see what it does to HEALTHY's
probabilities.

In [ ]:
# ✅ SOLUTION, warm-up 1: a preference changed by hand
my_preferences = start_preferences()    # a fresh copy of page 3's preferences
my_preferences[HEALTHY][REPLACE] = 5    # the preference for REPLACE in HEALTHY, set to 5
show_preferences(my_preferences)
print()
show_probabilities(probability_table(my_preferences))

In [ ]:
#@title 💡 Hint 1 for warm-up 1 (double-click to open) { display-mode: "form" }
# A preference of the table is my_preferences[state][action]: the row first, then the column. Give it a new
# value with =.

In [ ]:
#@title 💡 Hint 2 for warm-up 1 (double-click to open) { display-mode: "form" }
#     my_preferences[HEALTHY][REPLACE] = 5

In [ ]:
# Check warm-up 1
check_my_preferences(my_preferences)

### 🎯 Task D. The average return of a table of preferences

Write `average_return(preferences)`: the probability table of the preferences, `EPISODES` episodes of `WEEKS` weeks
with it, the $G_1$ of every episode, and their average in cents, as on page 3. Its first line is given:
`same_luck()` starts the luck again from its beginning, so every average plays the same episodes, and two tables of
preferences are compared on the same luck.

In [ ]:
# ✅ SOLUTION, task D: the average return of a table of preferences
def average_return(preferences):
    # preferences: a table of preferences.
    # Gives the average return of its policy over EPISODES episodes of WEEKS weeks, in cents.
    same_luck()                         # every average plays the same luck: the same episodes for every table
    policy = probability_table(preferences)            # the probability table of the preferences
    episodes = play_episodes(policy, WEEKS, EPISODES)  # EPISODES episodes of WEEKS weeks with the policy
    G1s = returns(episodes)                            # the G1 of every episode
    return round(average(G1s), 2)       # their average, in cents as on page 3

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# Each line calls one function you have: probability_table for the policy, play_episodes for the episodes,
# returns for their G1.
# play_episodes takes the policy, the weeks of an episode, then how many episodes: WEEKS and EPISODES.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
#     policy = probability_table(preferences)
#     episodes = play_episodes(policy, WEEKS, EPISODES)
#     G1s = returns(episodes)

In [ ]:
# Check task D
check_average_return(average_return)

In [ ]:
# The average return of page 3's preferences and of yours: run it twice
show_money("Page 3's preferences:", play(average_return, preferences))
show_money("Your preferences:    ", play(average_return, my_preferences))

**Reading the result.** Page 3's preferences give -\$172.24, the first point of page 3's learning curve, and the same
numbers come back every time the cell runs: every average replays the same luck. Your preferences replace a healthy
van in almost every week, at \$130 a time: -\$945.97.

## What you built

- **Many episodes**: `play_episodes` plays them, `returns` gives their $G_1$ and `average` their average.
- **The forward pass**: `average_return` gives the average return of 1,000 episodes of the policy of a table of
  preferences.

Next, part 3 builds the backward pass: the slope of a preference, its update and the whole loop of proto-REINFORCE:
[proto-REINFORCE, part 3: Old Bessie's backward pass](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_1c_old_bessie_backward_pass_solutions.ipynb).